# GW in the atomic limit

In GW the self-energy is given by

$\Sigma(\nu) \approx \frac{Un}{2} -\frac{1}{\beta}\sum_{\nu^\prime}G(\nu^\prime)\left[\frac{1}{4}\eta^D(\nu-\nu^\prime)+\frac{3}{4}\eta^M(\nu-\nu^\prime)\right]$

with the bosonic propagators 

$\eta^{D,M}(\omega)=\frac{\pm U}{1\mp UP(\omega)}$

and the polarization bubble

$P(\omega) = \frac{1}{\beta}\sum_\nu G(\omega+\nu)G(\nu)$.

Note that in the code we exclude constant asymptotic parts of the respective functions:

$\eta^{D,M}_\mathrm{code}(\omega)=\frac{\pm U}{1\mp UP(\omega)}\mp U = \frac{UP(\omega)}{1\mp UP(\omega)}$,

$\Sigma_\mathrm{code}(\nu)=\Sigma(\nu) - U n = - \frac{1}{\beta}\sum_{\nu^\prime}G(\nu^\prime)\left[\frac{1}{4}\eta_\mathrm{code}^D(\nu-\nu^\prime)+\frac{3}{4}\eta_\mathrm{code}^M(\nu-\nu^\prime)\right]$.

Here, $\Sigma_\mathrm{H} = U n = \frac{U}{\beta}\sum_{\nu^\prime}G(\nu^\prime)e^{i0^+\nu}$ is the Hartree term.



In [28]:
using MatsubaraFunctions
using NLsolve

As in the HF approximation, we define a function to compute the Matsubara sum over $G$:

In [29]:
function Matsubara_sum(f::MeshFunction)
    s = zero(eltype(f.data))
    νs = meshes(f,1)
    T  = temperature(νs)
    for i in eachindex(νs)
        s += f[i]
    end
    return T*s+0.5
end

Matsubara_sum (generic function with 1 method)

The structure "GWsolver" contains the parameters $T, U, N$, the MeshFunctions $\Sigma, G, P, \eta^{D,M}$ and symmetry groups. The Hartree term $\Sigma_\mathrm{H}$ is saved in an additional vector.

In [30]:
struct GWsolver
    T   :: Float64
    U   :: Float64
    N   :: Int64
    G   :: MeshFunction
    Σ   :: MeshFunction
    Σ_H :: Vector{ComplexF64}
    P   :: MeshFunction
    η_D :: MeshFunction
    η_M :: MeshFunction
    SGf :: SymmetryGroup
    SGb :: SymmetryGroup

    function GWsolver(T, U, N)

        # Hartree term
        Σ_H = [0.0]

        # fermionic containers
        gf = MatsubaraMesh(T,N,Fermion)
        G  = MeshFunction(gf; data_t = ComplexF64)
        Σ  = MeshFunction(gf; data_t = ComplexF64)

        # bosonic containers
        gb  = MatsubaraMesh(T,N,Boson)
        P   = MeshFunction(gb; data_t = ComplexF64)
        η_D = MeshFunction(gb; data_t = ComplexF64)
        η_M = MeshFunction(gb; data_t = ComplexF64)

        # symmetry groups
        # complex conjugation symmetry f(ω) = conj(f(-ω))
        conj_sym = Symmetry{1}() do args
            ω = args[1]
            ( -ω, ), Operation{ComplexF64}(sgn=false, con=true)
        end
        SGf = SymmetryGroup([conj_sym], G)
        SGb = SymmetryGroup([conj_sym], P)

        return new(T, U, N, G, Σ, Σ_H, P, η_D, η_M, SGf, SGb)
    end
end


For the fixed point solution, we define flattening for the self-energy $\Sigma_\mathrm{code}(\nu)$ MeshFunction and the Hartree term $\Sigma_\mathrm{H}$ into a vector $x$.

In [31]:
function flatten_Σ!(Σ::MeshFunction, Σ_H::Vector, x)
    N_Σ   = length(Σ.data)
    N_Σ_H = length(Σ_H)
    x[1:N_Σ]           .= @view Σ.data[:]
    x[N_Σ+1:N_Σ+N_Σ_H] .= @view Σ_H[:]
end

function unflatten_Σ!(Σ::MeshFunction, Σ_H::Vector, x)
    N_Σ   = length(Σ.data)
    N_Σ_H = length(Σ_H)
    Σ.data[:] .= x[1:N_Σ]
    Σ_H[:]    .= x[N_Σ+1:N_Σ+N_Σ_H]
end

unflatten_Σ! (generic function with 1 method)

Fixed point function for the self-consistency run over GWsolver.

In [32]:
function fixed_point!(F, x, S)

    # update Sigma
    unflatten_Σ!(S.Σ, S.Σ_H, x)

    # calculate G
    for i in eachindex(meshes(S.G,1))
        ν = value(value(points(meshes(S.G,1),i)))
        S.G[i] = 1.0 / (im * ν - S.Σ[i] - S.Σ_H[1])
    end

    sum_mesh = MatsubaraMesh(S.T, 4*S.N, Fermion)

    # calculate P using symmetries
    calc_P = InitFunction{1, ComplexF64}(
        x -> begin
            ω = x[1]
            P = zero(eltype(S.P.data))

            for i in eachindex(sum_mesh)
                ν = value(points(sum_mesh,i))
                P += S.G(ν + ω) * S.G(ν)
            end

            return S.T * P
        end
        )
    
    S.SGb(S.P, calc_P)

    # calculate η_D and η_M
    for i in meshes(S.η_D,1)
        S.η_D[i] = S.U * S.P[i] / (1.0 - S.U * S.P[i])
        S.η_M[i] = S.U * S.P[i] / (1.0 + S.U * S.P[i])
    end

    # calculate Σ using symmetries

    S.Σ_H[1] = S.U * Matsubara_sum(S.G)

    calc_Σ = InitFunction{1, ComplexF64}(
        x -> begin
            ν = x[1]
            Σ = zero(eltype(S.Σ.data))

            for i in eachindex(sum_mesh)
                νp = value(points(sum_mesh,i))
                Σ -= S.T * S.G(νp) * (
                    0.25 * S.η_D(ν - νp) +
                    0.75 * S.η_M(ν - νp))
            end

            return Σ
        end
        )

    S.SGf(S.Σ, calc_Σ)

    # calculate the residue
    flatten_Σ!(S.Σ, S.Σ_H, F)
    F .-= x

    return nothing
end

fixed_point! (generic function with 1 method)

In [33]:
T = 0.3
U = 0.9
N = 1000

S = GWsolver(T, U, N)
init   = zeros(eltype(S.G.data), length(S.G)+1)
result = nlsolve((F,x) -> fixed_point!(F, x, S), init, method =:anderson, m = 8, beta = 0.5, show_trace = true)

Iter     f(x) inf-norm    Step 2-norm 
------   --------------   --------------
     1     7.497662e-01              NaN
     2     2.515200e-01     3.725572e-02
     3     1.261368e-01     8.905085e-03
     4     1.839760e-02     1.750665e-04
     5     1.144247e-02     9.667463e-05
     6     3.967213e-03     8.633491e-06
     7     1.433675e-03     1.127823e-06
     8     8.838743e-05     4.196727e-09
     9     6.453638e-06     2.179312e-11
    10     5.642669e-07     1.833389e-13
    11     8.923312e-08     4.571879e-15
    12     1.731347e-09     1.752160e-18


Results of Nonlinear Solver Algorithm
 * Algorithm: Anderson m=8 beta=0.5 aa_start=1 droptol=1.0e10
 * Starting Point: ComplexF64[0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im,